# 00 — Profilage du fichier brut

Ce notebook décrit uniquement la source brute, sans correction. Il mesure la complétude, les doublons, les formats pays et codes postaux, et les anomalies visibles avant pipeline. Les invariants inter-étapes sont centralisés dans `99_validation.ipynb`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.normalize import normalize_country_code
from stages_cleaning.postal import normalize_postal_code, postal_code_status, zero_lost
from stages_cleaning.provenance import build_stage_id

from stages_cleaning.paths import dataset_paths

DATASET = dataset_paths(ROOT)
print("Jeu de données :", DATASET.name)
RAW_PATH = DATASET.raw
df = pd.read_csv(RAW_PATH, sep=";", dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA)
df.insert(0, "stage_id", build_stage_id(df))
print("Shape :", df.shape)
df.head()

## 1. Complétude

In [ ]:
missing = pd.DataFrame({
    "n_manquant": df.isna().sum(),
    "%_manquant": (df.isna().mean() * 100).round(2),
}).sort_values("%_manquant", ascending=False)
missing

## 2. Doublons exacts et logiques

In [ ]:
source_cols = [c for c in df.columns if c != "stage_id"]
print("Doublons exacts :", df.duplicated(subset=source_cols).sum())
logical_keys = ["email", "annee_academique", "type_stage", "entreprise"]
print("Lignes dans un groupe logique :", df.duplicated(subset=logical_keys, keep=False).sum())
df[df.duplicated(subset=source_cols, keep=False)].sort_values("email")

## 3. Cardinalités principales

In [ ]:
for col in ["programme", "annee_academique", "type_stage", "pays"]:
    print(f"=== {col} ({df[col].nunique()} valeurs distinctes) ===")
    print(df[col].value_counts(dropna=False).head(15), "\n")

## 4. Audit des pays source

In [ ]:
country_audit = pd.DataFrame({"pays_source": df["pays"], "pays_iso2": df["pays"].map(normalize_country_code)})
print("Valeurs source non vides :", country_audit["pays_source"].notna().sum())
print("Valeurs non interprétables :", (country_audit["pays_source"].notna() & country_audit["pays_iso2"].isna()).sum())
country_audit.drop_duplicates().sort_values("pays_source", na_position="last")

## 5. Audit des codes postaux source

In [ ]:
country_iso = df["pays"].map(normalize_country_code)
postal_norm = pd.Series([normalize_postal_code(cp, c) for cp, c in zip(df["code_postal"], country_iso)], index=df.index, dtype="string")
postal_status = pd.Series([postal_code_status(raw, norm, c, city) for raw, norm, c, city in zip(df["code_postal"], postal_norm, country_iso, df["ville"])], index=df.index, dtype="string")
print(postal_status.value_counts(dropna=False))
audit = df.loc[~postal_status.isin(["valid", "missing"]), ["stage_id", "entreprise", "adresse", "code_postal", "ville", "pays"]].copy()
audit["pays_iso2"] = country_iso[audit.index]
audit["code_postal_normalise"] = postal_norm[audit.index]
audit["statut"] = postal_status[audit.index]
audit.head(30)

## 6. Anomalies visibles dès le brut

In [ ]:
empty_loc = df[df[["entreprise", "adresse", "ville", "pays"]].isna().all(axis=1)]
print("Lignes sans entreprise/adresse/ville/pays :", len(empty_loc))
zero_mask = pd.Series([zero_lost(cp, p) for cp, p in zip(df["code_postal"], country_iso)], index=df.index)
print("Codes postaux avec zéro initial plausiblement perdu :", int(zero_mask.sum()))
non_efrei = df[~df["email"].str.endswith("@efrei.net", na=False)]
print("Emails hors @efrei.net :", len(non_efrei))
display(empty_loc[["stage_id", "etudiant", "email", "programme", "libelle_stage"]])